# Create Product and sales data

Continuing with other three csv files, my goal is to perform data transformation before combining into two files.

In [1]:
import duckdb

conn = duckdb.connect()

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaylimit = None

displaylimit: Value None will be treated as 0 (no limit)

In [2]:
%sql CREATE TABLE IF NOT EXISTS sales_details AS SELECT * FROM 'datasets/source_crm/sales_details.csv'
%sql CREATE TABLE IF NOT EXISTS prd_info AS SELECT * FROM 'datasets/source_crm/prd_info.csv'
%sql CREATE TABLE IF NOT EXISTS PX_CAT_G1V2 AS SELECT * FROM 'datasets/source_erp/PX_CAT_G1V2.csv'

Running query in 'duckdb'

Running query in 'duckdb'

Running query in 'duckdb'

Count
37


In [3]:
%%sql
-- Check if tables are loaded

SELECT
  table_name,
  column_name,
  data_type
FROM
  information_schema.columns

Running query in 'duckdb'

table_name,column_name,data_type
prd_info,prd_id,BIGINT
prd_info,prd_key,VARCHAR
prd_info,prd_nm,VARCHAR
prd_info,prd_cost,BIGINT
prd_info,prd_line,VARCHAR
prd_info,prd_start_dt,DATE
prd_info,prd_end_dt,DATE
PX_CAT_G1V2,ID,VARCHAR
PX_CAT_G1V2,CAT,VARCHAR
PX_CAT_G1V2,SUBCAT,VARCHAR


Once again, I must ensure ID and key columns are unique so they can form relations between each tables.

## Product Info

In [4]:
%%sql
SELECT
  *
FROM
  prd_info
LIMIT
  20;

Running query in 'duckdb'

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,None,R,2003-07-01,None
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,None,R,2003-07-01,None
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,None
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,None
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28


Since I am analysing current sales for this company, I can remove any historical data based on the product end date column.

In [5]:
%%sql
-- Check for duplicate product key of current sales

SELECT
  COUNT(*),
  COUNT(DISTINCT prd_key)
FROM
  prd_info
WHERE
  prd_end_dt IS NULL;

Running query in 'duckdb'

count_star(),count(DISTINCT prd_key)
197,197


In [6]:
%%sql
-- Split product key so category id can be extracted.

SELECT
  prd_id,
  TRIM(prd_key),
  REGEXP_EXTRACT (TRIM(prd_key), '([A-Z]*)-([A-Z]*)') AS CAT_ID,
  REPLACE (TRIM(prd_key), REGEXP_EXTRACT (TRIM(prd_key), '([A-Z]*)-([A-Z]*)-'),'') AS product_key
FROM
  prd_info
LIMIT
  20;

Running query in 'duckdb'

prd_id,"main.""trim""(prd_key)",CAT_ID,product_key
210,CO-RF-FR-R92B-58,CO-RF,FR-R92B-58
211,CO-RF-FR-R92R-58,CO-RF,FR-R92R-58
212,AC-HE-HL-U509-R,AC-HE,HL-U509-R
213,AC-HE-HL-U509-R,AC-HE,HL-U509-R
214,AC-HE-HL-U509-R,AC-HE,HL-U509-R
215,AC-HE-HL-U509,AC-HE,HL-U509
216,AC-HE-HL-U509,AC-HE,HL-U509
217,AC-HE-HL-U509,AC-HE,HL-U509
218,CL-SO-SO-B909-M,CL-SO,SO-B909-M
219,CL-SO-SO-B909-L,CL-SO,SO-B909-L


In [7]:
%%sql
-- Clean product line columns

SELECT
  CASE
    WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
    WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
    WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'Others'
    WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
    ELSE 'N/A'
  END AS prd_line
FROM
  prd_info
LIMIT
  5;

Running query in 'duckdb'

prd_line
Road
Road
Others
Others
Others


Now I can create a clean table of product info. 

In [8]:
%%sql
CREATE TABLE
  prd_info_new AS
SELECT
  prd_id,
  TRIM(prd_key) AS full_prd_key,
  REGEXP_EXTRACT (TRIM(prd_key), '([A-Z]*)-([A-Z]*)') AS CAT_ID,
  REPLACE (
    TRIM(prd_key),
    REGEXP_EXTRACT (TRIM(prd_key), '([A-Z]*)-([A-Z]*)-'),
    ''
  ) AS product_key,
  TRIM(prd_nm) AS prd_nm,
  COALESCE(prd_cost, 0) AS prd_cost,
  CASE
    WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
    WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
    WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'Others'
    WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
    ELSE 'N/A'
  END AS prd_line,
  prd_start_dt
FROM
  prd_info
WHERE
  prd_end_dt IS NULL;

Running query in 'duckdb'

Count
197


In [9]:
%%sql
-- Check clean table again

SELECT
  *
FROM
  prd_info_new
LIMIT
  10;

Running query in 'duckdb'

prd_id,full_prd_key,CAT_ID,product_key,prd_nm,prd_cost,prd_line,prd_start_dt
210,CO-RF-FR-R92B-58,CO-RF,FR-R92B-58,HL Road Frame - Black- 58,0,Road,2003-07-01
211,CO-RF-FR-R92R-58,CO-RF,FR-R92R-58,HL Road Frame - Red- 58,0,Road,2003-07-01
214,AC-HE-HL-U509-R,AC-HE,HL-U509-R,Sport-100 Helmet- Red,13,Others,2013-07-01
217,AC-HE-HL-U509,AC-HE,HL-U509,Sport-100 Helmet- Black,13,Others,2013-07-01
222,AC-HE-HL-U509-B,AC-HE,HL-U509-B,Sport-100 Helmet- Blue,13,Others,2013-07-01
225,CL-CA-CA-1098,CL-CA,CA-1098,AWC Logo Cap,7,Others,2013-07-01
228,CL-JE-LJ-0192-S,CL-JE,LJ-0192-S,Long-Sleeve Logo Jersey- S,38,Others,2013-07-01
231,CL-JE-LJ-0192-M,CL-JE,LJ-0192-M,Long-Sleeve Logo Jersey- M,38,Others,2013-07-01
234,CL-JE-LJ-0192-L,CL-JE,LJ-0192-L,Long-Sleeve Logo Jersey- L,38,Others,2013-07-01
237,CL-JE-LJ-0192-X,CL-JE,LJ-0192-X,Long-Sleeve Logo Jersey- XL,38,Others,2013-07-01


## Product Category

In [10]:
%%sql
SELECT
  *
FROM
  PX_CAT_G1V2
LIMIT
  5;

Running query in 'duckdb'

ID,CAT,SUBCAT,MAINTENANCE
AC_BR,Accessories,Bike Racks,True
AC_BS,Accessories,Bike Stands,False
AC_BC,Accessories,Bottles and Cages,False
AC_CL,Accessories,Cleaners,True
AC_FE,Accessories,Fenders,False


In [11]:
%%sql
-- Check for duplicate ID

SELECT
  COUNT(*),
  COUNT(DISTINCT ID)
FROM
  PX_CAT_G1V2;

Running query in 'duckdb'

count_star(),count(DISTINCT ID)
37,37


Now I create another clean table, making sure the final data transformation process is present in my code.

In [12]:
%%sql
CREATE TABLE
  prd_cat AS
SELECT
  REPLACE (TRIM(ID), '_', '-') AS ID,
  TRIM(CAT) AS CAT,
  TRIM(SUBCAT) AS SUBCAT,
  MAINTENANCE
FROM
  PX_CAT_G1V2;

Running query in 'duckdb'

Count
37


## Sales Details

In [13]:
%%sql
SELECT *
FROM sales_details
LIMIT 5;

Running query in 'duckdb'

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,20101229,20110105,20110110,3578,1,3578
SO43698,BK-M82S-44,28389,20101229,20110105,20110110,3400,1,3400
SO43699,BK-M82S-44,25863,20101229,20110105,20110110,3400,1,3400
SO43700,BK-R50B-62,14501,20101229,20110105,20110110,699,1,699
SO43701,BK-M82S-44,11003,20101229,20110105,20110110,3400,1,3400


In [14]:
%%sql
--Check sales value

SELECT sls_sales
FROM sales_details
ORDER BY sls_sales
LIMIT 10;

Running query in 'duckdb'

sls_sales
-54
-35
-18
0
0
2
2
2
2
2


In [15]:
%%sql
-- Now check price value

SELECT sls_price
FROM sales_details
ORDER BY sls_price
LIMIT 10;

Running query in 'duckdb'

sls_price
-1701
-769
-30
-22
-21
2
2
2
2
2


In [16]:
%%sql
-- Check total sales relative to other columns

SELECT
  sls_ord_num,
  sls_prd_key,
  sls_cust_id,
  sls_sales,
  sls_quantity,
  sls_price
FROM
  sales_details
WHERE
  sls_sales <= 0

Running query in 'duckdb'

sls_ord_num,sls_prd_key,sls_cust_id,sls_sales,sls_quantity,sls_price
SO61570,CA-1098,17809,-18,1,9
SO61636,BC-M005,12914,0,1,10
SO68889,TT-R982,14086,0,1,4
SO69066,SJ-0194-L,17923,-54,1,54
SO69215,TI-M823,16864,-35,1,35


In [17]:
%%sql
-- Check sales price relative to other columns

SELECT
  sls_ord_num,
  sls_prd_key,
  sls_cust_id,
  sls_sales,
  sls_quantity,
  sls_price
FROM
  sales_details
WHERE
  sls_price <= 0

Running query in 'duckdb'

sls_ord_num,sls_prd_key,sls_cust_id,sls_sales,sls_quantity,sls_price
SO57804,BK-M38S-40,16470,769,1,-769
SO57916,TI-M602,23024,30,1,-30
SO58326,FE-6654,12045,22,1,-22
SO58623,BK-R79Y-44,17012,1701,1,-1701
SO58818,TI-R092,21387,21,1,-21


Notice that any negative sales price is just data entry error and removing the negative sign solves the issue. Total sales can also be obtained by multiplying sales price and quantity. I will do one of the cleaning process for the new cell.

In [18]:
%%sql
SELECT
  sls_price,
  CASE
    WHEN sls_price < 0 THEN sls_price * -1
    ELSE sls_price
  END AS correct_price
FROM
  sales_details
WHERE
  sls_price <= 0

Running query in 'duckdb'

sls_price,correct_price
-769,769
-30,30
-22,22
-1701,1701
-21,21


In [19]:
%%sql
-- Check all date columns

SELECT
    sls_order_dt,
    STRPTIME(CAST(sls_order_dt AS varchar), '%Y%m%d') AS sls_order_dt_new,
    sls_ship_dt,
    STRPTIME(CAST(sls_ship_dt AS varchar), '%Y%m%d') AS sls_ship_dt_new,
    sls_due_dt,
    STRPTIME(CAST(sls_due_dt AS varchar), '%Y%m%d') AS sls_due_dt_new,
FROM sales_details
LIMIT 5;

Running query in 'duckdb'

sls_order_dt,sls_order_dt_new,sls_ship_dt,sls_ship_dt_new,sls_due_dt,sls_due_dt_new
20101229,2010-12-29 00:00:00,20110105,2011-01-05 00:00:00,20110110,2011-01-10 00:00:00
20101229,2010-12-29 00:00:00,20110105,2011-01-05 00:00:00,20110110,2011-01-10 00:00:00
20101229,2010-12-29 00:00:00,20110105,2011-01-05 00:00:00,20110110,2011-01-10 00:00:00
20101229,2010-12-29 00:00:00,20110105,2011-01-05 00:00:00,20110110,2011-01-10 00:00:00
20101229,2010-12-29 00:00:00,20110105,2011-01-05 00:00:00,20110110,2011-01-10 00:00:00


In [20]:
%%sql
-- Check for empty date

SELECT sls_order_dt
FROM sales_details
ORDER BY sls_order_dt
LIMIT 5;

Running query in 'duckdb'

sls_order_dt
0
0
0
0
0


In [21]:
%%sql
-- Check for invalid date

SELECT sls_order_dt
FROM sales_details
WHERE sls_order_dt != 0
ORDER BY sls_order_dt
LIMIT 5;

Running query in 'duckdb'

sls_order_dt
5489
32154
20101229
20101229
20101229


The simple solution for the date columns is to remove invalid dates then convert to proper date format. With that, I can create another clean table, remembering to derive total sales values by multiplying sales price and quantity.

In [22]:
%%sql

CREATE TABLE total_sales AS
SELECT 
    TRIM(sls_ord_num) AS order_number,
    TRIM(sls_prd_key) AS product_key,
    sls_cust_id AS customer_id,
    CASE
        WHEN sls_order_dt = 0 OR LEN(CAST(sls_order_dt AS varchar)) != 8 THEN NULL
        ELSE STRPTIME(CAST(sls_order_dt AS varchar), '%Y%m%d')
    END AS order_date,
    CASE
        WHEN sls_ship_dt = 0 OR LEN(CAST(sls_ship_dt AS varchar)) != 8 THEN NULL
        ELSE STRPTIME(CAST(sls_ship_dt AS varchar), '%Y%m%d')
    END AS shipping_date,
    CASE
        WHEN sls_due_dt = 0 OR LEN(CAST(sls_due_dt AS varchar)) != 8 THEN NULL
        ELSE STRPTIME(CAST(sls_due_dt AS varchar), '%Y%m%d')
    END AS due_date,
    CASE
        WHEN sls_price < 0 THEN sls_price * -1
        ELSE sls_price
    END AS price,
    sls_quantity AS quantity,
    sls_quantity*price AS sales_amount
FROM sales_details;

Running query in 'duckdb'

Count
60398


In [23]:
%%sql
-- Check final table
    
SELECT *
FROM total_sales
LIMIT 10;

Running query in 'duckdb'

order_number,product_key,customer_id,order_date,shipping_date,due_date,price,quantity,sales_amount
SO43697,BK-R93R-62,21768,2010-12-29 00:00:00,2011-01-05 00:00:00,2011-01-10 00:00:00,3578,1,3578
SO43698,BK-M82S-44,28389,2010-12-29 00:00:00,2011-01-05 00:00:00,2011-01-10 00:00:00,3400,1,3400
SO43699,BK-M82S-44,25863,2010-12-29 00:00:00,2011-01-05 00:00:00,2011-01-10 00:00:00,3400,1,3400
SO43700,BK-R50B-62,14501,2010-12-29 00:00:00,2011-01-05 00:00:00,2011-01-10 00:00:00,699,1,699
SO43701,BK-M82S-44,11003,2010-12-29 00:00:00,2011-01-05 00:00:00,2011-01-10 00:00:00,3400,1,3400
SO43702,BK-R93R-44,27645,2010-12-30 00:00:00,2011-01-06 00:00:00,2011-01-11 00:00:00,3578,1,3578
SO43703,BK-R93R-62,16624,2010-12-30 00:00:00,2011-01-06 00:00:00,2011-01-11 00:00:00,3578,1,3578
SO43704,BK-M82B-48,11005,2010-12-30 00:00:00,2011-01-06 00:00:00,2011-01-11 00:00:00,3375,1,3375
SO43705,BK-M82S-38,11011,2010-12-30 00:00:00,2011-01-06 00:00:00,2011-01-11 00:00:00,3400,1,3400
SO43706,BK-R93R-48,27621,2010-12-31 00:00:00,2011-01-07 00:00:00,2011-01-12 00:00:00,3578,1,3578


## Final results

In [24]:
%%sql
-- Check clean tables again
    
SELECT
  table_name,
  column_name,
  data_type
FROM
  information_schema.columns
WHERE table_name IN ('prd_info_new', 'prd_cat', 'total_sales')

Running query in 'duckdb'

table_name,column_name,data_type
prd_cat,ID,VARCHAR
prd_cat,CAT,VARCHAR
prd_cat,SUBCAT,VARCHAR
prd_cat,MAINTENANCE,BOOLEAN
prd_info_new,prd_id,BIGINT
prd_info_new,full_prd_key,VARCHAR
prd_info_new,CAT_ID,VARCHAR
prd_info_new,product_key,VARCHAR
prd_info_new,prd_nm,VARCHAR
prd_info_new,prd_cost,BIGINT


In [25]:
%%sql
-- Test if product info table and category table merges correctly
    
SELECT *
FROM prd_info_new
LEFT JOIN prd_cat
ON prd_info_new.CAT_ID = prd_cat.ID
LIMIT 5;

Running query in 'duckdb'

prd_id,full_prd_key,CAT_ID,product_key,prd_nm,prd_cost,prd_line,prd_start_dt,ID,CAT,SUBCAT,MAINTENANCE
210,CO-RF-FR-R92B-58,CO-RF,FR-R92B-58,HL Road Frame - Black- 58,0,Road,2003-07-01,CO-RF,Components,Road Frames,True
211,CO-RF-FR-R92R-58,CO-RF,FR-R92R-58,HL Road Frame - Red- 58,0,Road,2003-07-01,CO-RF,Components,Road Frames,True
214,AC-HE-HL-U509-R,AC-HE,HL-U509-R,Sport-100 Helmet- Red,13,Others,2013-07-01,AC-HE,Accessories,Helmets,True
217,AC-HE-HL-U509,AC-HE,HL-U509,Sport-100 Helmet- Black,13,Others,2013-07-01,AC-HE,Accessories,Helmets,True
222,AC-HE-HL-U509-B,AC-HE,HL-U509-B,Sport-100 Helmet- Blue,13,Others,2013-07-01,AC-HE,Accessories,Helmets,True


In [26]:
%%sql
--Now perform merging of two tables
    
CREATE TABLE final_product AS
SELECT 
    product_key,
    prd_id AS product_id,
    full_prd_key AS product_number,
    prd_nm AS product_name,
    CAT_ID AS category_id,
    CAT AS category,
    SUBCAT AS subcategory,
    MAINTENANCE AS maintenance,
    prd_cost AS cost,
    prd_line AS product_line,
    prd_start_dt AS start_date
FROM prd_info_new
LEFT JOIN prd_cat
ON prd_info_new.CAT_ID = prd_cat.ID;

Running query in 'duckdb'

Count
197


In [27]:
%sql COPY final_product TO 'datasets/final_product.csv' (HEADER, DELIMITER ',');

%sql COPY total_sales TO 'datasets/total_sales.csv' (HEADER, DELIMITER ',');

Running query in 'duckdb'

Running query in 'duckdb'

Count
60398


In [28]:
# Done
conn.close()